# PIG Pipeline Walkthrough

This notebook demonstrates the full **Patch-Influence Graph (PIG)** pipeline step by step,
from raw patch effects to causal evaluation.

| Stage | Input → Output |
|---|---|
| **0. Alignment baseline** | Two effect matrices → diff graph → always aligned / always not-aligned |
| **1. Input** | Prompt pairs → `PatchEffectTensor` per example |
| **2. Effect matrix** | Stack tensors → `[N, D]` |
| **3. Correlation** | Effect matrix → `[D, D]` co-influence matrix |
| **4. Graph** | Correlation + constraints + top-k → `PatchInfluenceGraph` |
| **5. WL Embedding** | Graph → fixed-length feature vector |
| **6. Candidate proposal** | Correlations → ranked directed edge list |
| **7. Causal evaluation** | Candidates + model → Level A / B / C metrics |


## TL;DR (minimal, non-technical)

If you only remember one thing, remember this loop:

1. Generate **clean vs corrupted** prompt pairs.
2. Patch one internal node at a time and measure target-score recovery.
3. Stack those recoveries into a matrix across examples.
4. Turn co-varying nodes into a sparse directed graph.
5. Embed graphs (WL) and compare kernels (classical/quantum).
6. Optionally test top edges with causal diagnostics (Levels A/B/C).

This notebook keeps each step tiny and explicit so every transition is visible.


## Glossary (8 terms)

- **PromptPair**: one `(x_cln, x_crp, y_star)` example.
- **PatchEffectTensor**: per-example effects over nodes `[L, T, C]`.
- **Node `(l,t,c)`**: layer, token position, component type.
- **Slice**: a task/corruption subset (e.g., `name_swap`, `abba`).
- **Effect matrix `[N, D]`**: flattened per-example effects.
- **Correlation matrix `[D, D]`**: co-variation between nodes.
- **PatchInfluenceGraph**: sparse directed graph from correlation + constraints.
- **WL features**: fixed-length graph vector used by kernel classifiers.


In [ ]:
import pytest
import numpy as np
import matplotlib.pyplot as plt

from pig.patching import ComponentSpec, PatchEffectDataset, PatchEffectTensor
from pig.prompts import PromptPair, SliceLabel
from pig.graph import GraphBuilder
from pig.embeddings import WLEncoder, compute_wl_features
from pig.graphs.diff_correlation_topk import DiffCorrelationGraphBuilder
from pig.causal import (
    CausalEdgeCandidate, CausalEvalConfig,
    propose_causal_edge_candidates, evaluate_causal_edges,
    restoration_fraction, mediation_score, necessity_score,
    activation_influence_score,
)
from pig.model import create_model
from pig.prompts import create_ioi_dataset

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

def show_matrix(name, matrix, row_labels=None, col_labels=None):
    matrix = np.asarray(matrix)
    print(f"\n{name} shape={matrix.shape}")

    if row_labels is None and col_labels is None:
        print(np.array2string(matrix, precision=3, suppress_small=True, floatmode="fixed"))
        return

    if row_labels is None:
        row_labels = [f"r{i}" for i in range(matrix.shape[0])]
    if col_labels is None:
        col_labels = [f"c{j}" for j in range(matrix.shape[1])]

    header = "".ljust(8) + " ".join(f"{c:>8}" for c in col_labels)
    print(header)
    for label, row in zip(row_labels, matrix):
        row_text = " ".join(f"{v:8.3f}" for v in row)
        print(f"{label:>8} {row_text}")


---
## Stage 0 — Alignment baseline: guaranteed aligned / not-aligned

The **diff graph** (`DiffCorrelationGraphBuilder`) captures how the fine-tuned model
differs from the base model. It builds a correlation matrix over the **per-example diff**:

$$\mathbf{d}^{(i)} = \mathbf{e}^{(i)}_{\text{ft}} - \mathbf{e}^{(i)}_{\text{base}}$$

The correlation is only non-zero if the diff has **variance across examples**.  
This gives us two matrices with guaranteed outcomes:

| Construction | Diff | Variance | Edges | Interpretation |
|---|---|---|---|---|
| `ft[i] = base[i]` | 0 | 0 | 0 | **Always aligned** — models are identical |
| `ft[i] = base[i] + s_i × direction` | `s_i × direction` | > 0 | correlated | **Always not-aligned** — consistent structural change |

In [ ]:
N, L, T, C = 8, 3, 3, 1
sl = SliceLabel(task="ioi", corruption="name_swap")
component_axis = [ComponentSpec(node_type="res")]

# Shared base effects (same prompt pairs for both models)
base_eff = [rng.standard_normal((L, T, C)).astype(np.float32) for _ in range(N)]

def make_dataset(effects_fn):
    ds = PatchEffectDataset()
    for i in range(N):
        pair = PromptPair(x_cln=f"cln-{i}", x_crp=f"crp-{i}",
                          y_star="tgt", slice_label=sl, meta={})
        ds.add(PatchEffectTensor(effects=effects_fn(i), component_axis=component_axis,
                                  prompt_pair=pair, base_score=-1.0, clean_score=1.0))
    return ds

ds_base = make_dataset(lambda i: base_eff[i])

# ALWAYS ALIGNED: ft == base  →  diff = 0 for every example
ds_aligned = make_dataset(lambda i: base_eff[i].copy())

# ALWAYS NOT-ALIGNED: ft = base + s_i × direction
#   s_i is a random scalar (introduces variance across examples)
#   direction is a fixed vector (introduces correlation between specific nodes)
direction = np.zeros((L, T, C), dtype=np.float32)
direction[1, :, 0] = 1.0            # layer 1, all tokens, fired consistently
scales = rng.standard_normal(N).astype(np.float32)   # random strength per example
ds_not_aligned = make_dataset(lambda i: base_eff[i] + scales[i] * direction)

print("Base dataset:",     len(ds_base), "tensors")
print("Aligned dataset:",  len(ds_aligned), "tensors  (ft == base)")
print("Not-aligned:",      len(ds_not_aligned), "tensors  (ft = base + s_i × direction)")

In [ ]:
diff_builder = DiffCorrelationGraphBuilder(base_dataset=ds_base, k=2, enforce_direction=True)

g_aligned     = diff_builder.build_from_slice(ds_aligned,     sl)
g_not_aligned = diff_builder.build_from_slice(ds_not_aligned, sl)

print("=== ALIGNED (ft == base, diff = 0) ===")
print(f"  Edges: {g_aligned.num_edges}")
print(f"  ⟹ Zero variance in diff matrix → no correlation → no edges")
print()
print("=== NOT ALIGNED (ft = base + s_i × direction) ===")
print(f"  Edges: {g_not_aligned.num_edges}")
node_labels_33 = [f"L{l}T{t}" for l in range(L) for t in range(T)]
for e in sorted(g_not_aligned.edges, key=lambda e: -abs(e.weight)):
    s, d = g_not_aligned.nodes[e.src], g_not_aligned.nodes[e.dst]
    print(f"  L{s.layer}T{s.token} → L{d.layer}T{d.token}   w={e.weight:+.3f}")
print(f"  ⟹ All edges in layer 1 (where direction fires), C = 1.0")

In [ ]:
# Show the diff correlation matrices
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
nl = node_labels_33
adj_aligned = g_aligned.get_adjacency_matrix()
adj_not_aligned = g_not_aligned.get_adjacency_matrix()

for ax, adj, title in zip(axes,
                          [adj_aligned, adj_not_aligned],
                          ["Aligned (diff = 0)\n-> no edges",
                           "Not-aligned (ft = base + s_i x direction)\n-> C = 1.0 at layer 1"]):
    im = ax.imshow(adj, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(L*T)); ax.set_xticklabels(nl, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(L*T)); ax.set_yticklabels(nl, fontsize=8)
    ax.set_title(title)
    plt.colorbar(im, ax=ax)

plt.suptitle("Diff graph adjacency matrices", y=1.02)
plt.tight_layout()
plt.show()

show_matrix("Adjacency matrix (aligned)", adj_aligned, nl, nl)
show_matrix("Adjacency matrix (not aligned)", adj_not_aligned, nl, nl)


In [ ]:
# WL features — compute jointly to share vocabulary
sl_a = SliceLabel(task="demo", corruption="aligned")
sl_n = SliceLabel(task="demo", corruption="not_aligned")
g_aligned.slice_label     = sl_a
g_not_aligned.slice_label = sl_n

fm = compute_wl_features({sl_a: g_aligned, sl_n: g_not_aligned}, depth=2)
X_diff = fm.to_matrix()   # [2, vocab]

print(f"WL feature matrix: {X_diff.shape}  (2 graphs × {fm.num_features} features)")
print(f"L1 distance between aligned and not-aligned: {np.sum(np.abs(X_diff[0] - X_diff[1])):.0f}")
print()
print("Features that differ (aligned vs not-aligned):")
for j in range(fm.num_features):
    if X_diff[0, j] != X_diff[1, j]:
        depth = fm.vocabulary[j][:2]
        print(f"  {depth}  aligned={X_diff[0,j]:.0f}  not_aligned={X_diff[1,j]:.0f}")

print()
print("⟹ The diff graph's edge structure is detectable from depth-1+ WL labels.")

**Why this is guaranteed:**

- **Aligned** (`ft = base`): the diff matrix `[N, D]` is all zeros → mean = 0, std = 0 → normalized = 0/1 = 0 → correlation C = 0 everywhere → no edges pass the `abs(weight) > 0` threshold.

- **Not-aligned** (`ft = base + s_i × direction`): the diff at node $j$ is $s_i \cdot \text{direction}_j$. Nodes where `direction ≠ 0` (layer 1) have diff proportional to $s_i$ → perfectly correlated → $C = 1.0$ → all layer-1 edges appear at maximum weight.

This means the **structure of the input matrix directly controls the diff graph**, with no randomness involved.

---
## Stage 1 — Input: PatchEffectTensor

A `PatchEffectTensor` stores, for a single prompt pair $(x^\text{cln}, x^\text{crp}, y^*)$,
the **patch effect** at every node $(l, t, c)$:

$$E^{(i)}_{l,t,c} = O_{\text{patch}}(\{(l,t,c)\}) - O_{\text{base}}$$

- `effects` shape: `[num_layers, num_tokens, num_components]` — dtype `float32`
- `base_score`: log-prob on corrupted prompt (before patching)
- `clean_score`: log-prob on clean prompt

We create **8 examples** with a planted causal signal:
node `(L=0, T=0)` and node `(L=1, T=1)` share a common random signal,
so they will have the highest pairwise correlation.

In [ ]:
slice_label = SliceLabel(task="demo", corruption="word_swap")

# Background noise
effects_all = rng.standard_normal((N, L, T, C)).astype(np.float32) * 0.1

# Plant a causal signal: (L=0,T=0) → (L=1,T=1)
signal = rng.standard_normal(N).astype(np.float32)
effects_all[:, 0, 0, 0] += signal          # source: strong signal
effects_all[:, 1, 1, 0] += 0.9 * signal    # destination: correlated signal

tensors = []
for i in range(N):
    pair = PromptPair(x_cln=f"clean-{i}", x_crp=f"corrupt-{i}",
                      y_star="target", slice_label=slice_label, meta={})
    tensors.append(PatchEffectTensor(
        effects=effects_all[i],
        component_axis=component_axis,
        prompt_pair=pair,
        base_score=-1.0,
        clean_score=1.0,
    ))

print(f"Tensor shape:  {tensors[0].effects.shape}  (L, T, C)")
print(f"Examples:      {N}")

---
## Stage 2 — Effect Matrix `[N, D]`

Flatten each tensor and normalize by the per-example performance delta:

$$\mathbf{X}_{i,j} = \frac{\text{effects}_{i,j}}{\max(O^\text{cln}_i - O^\text{base}_i,\ \varepsilon)}$$

Result: an `[N, D]` matrix where `D = L × T × C = 9`.

In [ ]:
eps = 1e-6

rows = [(t.effects / max(t.clean_score - t.base_score, eps)).reshape(-1)
        for t in tensors]
X = np.stack(rows, axis=0)   # shape [N, D]
D = X.shape[1]

node_labels = [f"L{l}T{t}" for l in range(L) for t in range(T)]
example_labels = [f"ex{i}" for i in range(N)]
print(f"Effect matrix: {X.shape}  (N={N}, D={D})")
show_matrix("Effect matrix X", X, example_labels, node_labels)


---
## Stage 3 — Correlation Matrix `[D, D]`

Center and standardize each column, then compute:

$$\mathbf{C} = \frac{\tilde{\mathbf{X}}^\top \tilde{\mathbf{X}}}{N}$$

$C_{ij}$ = correlation of patch-effect profiles between node $i$ and $j$.  
High $|C_{ij}|$ means these two nodes tend to co-activate across examples.

In [ ]:
X_centered = X - X.mean(axis=0, keepdims=True)
std = np.std(X_centered, axis=0, keepdims=True)
std = np.where(std < 1e-8, 1.0, std)
X_norm = X_centered / std

C_matrix = (X_norm.T @ X_norm) / N
np.fill_diagonal(C_matrix, 0.0)

abs_C = np.abs(C_matrix.copy()); np.fill_diagonal(abs_C, 0.0)
si, di = np.unravel_index(np.argmax(abs_C), abs_C.shape)
print(f"Strongest pair: {node_labels[si]} <-> {node_labels[di]}  C = {C_matrix[si, di]:.3f}")
show_matrix("Correlation matrix C", C_matrix, node_labels, node_labels)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(C_matrix, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(D)); ax.set_xticklabels(node_labels, rotation=45, ha="right")
ax.set_yticks(range(D)); ax.set_yticklabels(node_labels)
ax.set_title("Correlation matrix C")
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()


---
## Stage 4 — Build `PatchInfluenceGraph`

Two constraints applied to `C` before creating edges:

1. **Direction constraint**: keep only $u \to v$ where $u < v$  
2. **Top-k sparsification**: at most `k` outgoing edges per source node

In [ ]:
dataset = PatchEffectDataset()
for t in tensors:
    dataset.add(t)

builder = GraphBuilder(k=2, enforce_direction=True)
graph = builder.build_from_slice(dataset, slice_label)

print(f"Nodes: {graph.num_nodes}  Edges: {graph.num_edges}")
print()
for e in sorted(graph.edges, key=lambda e: -abs(e.weight)):
    s, d = graph.nodes[e.src], graph.nodes[e.dst]
    print(f"  L{s.layer}T{s.token} -> L{d.layer}T{d.token}   w={e.weight:+.3f}")

graph_adj = graph.get_adjacency_matrix()
show_matrix("PatchInfluenceGraph adjacency", graph_adj, node_labels, node_labels)


---
## Stage 5 — WL Graph Embedding

Iteratively hash node neighborhoods to produce a fixed-length feature vector:

1. **Depth 0**: each node → label `L{l}_T{t}_res`
2. **Depth d**: new label = `hash(own_label | sorted_neighbor_labels)`
3. Count label occurrences at each depth → feature counter

In [ ]:
feature_matrix = compute_wl_features({slice_label: graph}, depth=2)
X_wl = feature_matrix.to_matrix()

print(f"WL feature matrix: {X_wl.shape}  (graphs × features)")

---
## Stage 6 — Propose Causal Edge Candidates

`propose_causal_edge_candidates` wraps Stages 2–4, returning top-k directed edges
sorted by $|C_{ij}|$.

In [ ]:
candidates = propose_causal_edge_candidates(
    tensors, num_edges=5, node_types=["res"], enforce_direction=True,
)

print(f"{'rank':>4}  {'edge_id':<30}  {'corr':>6}")
print("-" * 44)
for c in candidates:
    print(f"{c.rank:>4}  {c.edge_id():<30}  {c.correlation_score:>+.3f}")

assert candidates[0].edge_id() == "L0T0.res->L1T1.res"
print("\nRank 1 is the planted L0T0 → L1T1 edge. ✓")

---
## Stage 7 — Causal Evaluation: Levels A / B / C

| Level | Metric | Formula | Meaning |
|---|---|---|---|
| **A** | $I(u{\to}v)$ | $\|a_v^{\text{patch}(u)} - a_v^{\text{base}}\|_2 / \|a_v^{\text{cln}} - a_v^{\text{base}}\|_2$ | Does patching $u$ displace $v$? |
| **B** | $M(u{\to}v)$ | $R_{uv} - R_v$ | Does $u$'s effect flow through $v$? |
| **C** | $\text{nec}(u{\to}v)$ | $R_u - R_{u,\text{clamp}(v)}$ | Is $v$ necessary for $u$'s effect? |

In [ ]:
model = create_model(model_name="toy_transformer", device="cpu")
pairs = create_ioi_dataset(n_examples=4, corruption="name_swap", seed=7)

eval_candidate = CausalEdgeCandidate(
    src_layer=0, src_token=0, src_node_type="att", src_head=0,
    dst_layer=1, dst_token=1, dst_node_type="att", dst_head=0,
)
config = CausalEvalConfig(eps=1e-6, bootstrap_samples=100, permutation_samples=100, seed=42)

result = evaluate_causal_edges(model, pairs, [eval_candidate], config)
edge = result.edges[0]

def fmt(section, key):
    s = edge[section][key]
    return (f"mean={s['mean']:+.3f}  95%CI[{s['ci_low']:+.3f},{s['ci_high']:+.3f}]  "
            f"p={s['p_value']:.3f}  p_fdr={s.get('p_value_fdr_bh', float('nan')):.3f}")

print(f"Edge: {edge['edge_id']}\n")
print(f"Level A — I(u→v):   {fmt('level_a', 'I')}")
print(f"Level B — M(u→v):   {fmt('level_b', 'M')}")
print(f"Level C — nec(u→v): {fmt('level_c', 'necessity')}")
print(f"\nClassification: {edge['diagnostics']['classification']}")

---
## Appendix — Formula sanity check

In [ ]:
r_u  = restoration_fraction(12.0, base_score=10.0, clean_score=14.0)  # 0.50
r_v  = restoration_fraction(11.0, base_score=10.0, clean_score=14.0)  # 0.25
r_uv = restoration_fraction(13.0, base_score=10.0, clean_score=14.0)  # 0.75
assert r_u == pytest.approx(0.50)

M = mediation_score(r_uv, r_v)
assert M == pytest.approx(0.50)

r_u_clamp = restoration_fraction(10.5, base_score=10.0, clean_score=14.0)
nec = necessity_score(r_u, r_u_clamp)

I = activation_influence_score(
    np.array([1.0, 0.0]), np.array([0.0, 0.0]), np.array([2.0, 0.0])
)
assert I == pytest.approx(0.50)

print(f"R_u={r_u:.2f}  R_v={r_v:.2f}  R_uv={r_uv:.2f}")
print(f"M={M:.2f}  nec={nec:.3f}  I={I:.2f}")
print("All assertions pass.")

## One Real Command (end-to-end)

Notebook stages are conceptual. The production run is one command:

```bash
uv run pig pipeline --model-name toy_transformer
```

`pig pipeline` executes these scripts in order (`src/pig/cli.py`, `VALIDATION_SEQUENCE`):

- Stage 1 (prompt pairs): `validate_story_1_2.py`
- Stage 1-2 (patching/effects): `validate_story_2_1.py`, `validate_story_2_2.py`
- Stage 3-4 (graph build): `validate_story_3_1.py`
- Stage 5 (WL embedding): `validate_story_4_1.py`
- Classical kernel baseline: `validate_story_4_2.py`
- Quantum baselines: `validate_story_5_1.py`, `validate_story_5_2.py`
- Figures/artifacts: `generate_pipeline_artifacts.py`

So this notebook is the **minimal mental model**, and `pig pipeline` is the **full operational execution**.


---
## Summary

```
Base effects + FT effects  (same prompt pairs)
         │
         ▼  diff[i] = ft[i] - base[i]  per example
         │
         ├── diff = 0  (ft == base)      → no variance → C = 0 → 0 edges  → ALIGNED
         └── diff = sᵢ × direction       → variance + correlation → edges  → NOT ALIGNED

PatchEffectTensor [L, T, C]  ×N
         │
         ▼  flatten + normalize by (clean − base)
Effect matrix  [N, D]    D = L×T×C
         │
         ▼  centre, standardise → C = X̃ᵀX̃ / N
Correlation    [D, D]
         │
         ▼  direction constraint + top-k
PatchInfluenceGraph  (nodes, sparse edges, weights)
         │
         ├─▶  WL Embedding → [vocab_size] feature vector
         └─▶  top-k candidates → CausalEvalResult
                  Level A: I(u→v)    — activation displacement
                  Level B: M(u→v)    — mediation
                  Level C: nec(u→v)  — necessity
```
